# CAIRLab Hackathon — Day 1: Beginner Track

**The story:** You are a consortium of five banks. You all want to catch the same fraud and
network-intrusion patterns, but you are legally forbidden from pooling your customer traffic
data into one place. Federated learning (FL) is the answer: each bank trains a model **locally**
on its own data, and only the trained *model updates* — never the raw data — are shared and
averaged into one global model.

**What this notebook gives you:**
- A cleaned, pre-partitioned intrusion-detection dataset (NSL-KDD), split evenly across 5
  simulated clients ("banks").
- A **working, weak baseline model** and a **working FedAvg training loop** that already runs
  end to end and evaluates on a held-out test set.
- One track: build the best **local** detection model. You do **not** need to build any of
  the FL plumbing from scratch — your job is to improve what's here.

**Today is Day 1.** Two more tracks unlock in a follow-up notebook on Day 2 — you'll get it
then, along with instructions for how it builds on today's work. Nothing about them is in
this notebook, so there's no need to go looking.

**Run every cell top to bottom once**, then work in the `# 🔧 YOUR TURN` section near the end.

**Scoring today:** your final model is scored automatically on a real Kaggle leaderboard
(link from the organizers) against a held-out test set you never see — not self-reported.


## 0. Setup

In [ ]:

!pip -q install torch scikit-learn pandas numpy --upgrade 2>/dev/null
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import copy, json, os
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import precision_score, recall_score, f1_score

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

os.makedirs("data", exist_ok=True)
print("Setup complete.")


## 1. Load and clean the dataset (NSL-KDD)

NSL-KDD is a well-documented network intrusion dataset. Each row is a network connection
record; the label says whether it's `normal` traffic or one of several attack types. We
collapse this into **binary** labels here (`0 = normal`, `1 = attack`).

We download the standard train/test split directly from a public GitHub mirror so this cell
works on a fresh Colab or Kaggle Notebook runtime with no manual upload.


In [ ]:

TRAIN_URL = "https://raw.githubusercontent.com/jmnwong/NSL-KDD-Dataset/master/KDDTrain%2B.txt"
TEST_URL  = "https://raw.githubusercontent.com/jmnwong/NSL-KDD-Dataset/master/KDDTest%2B.txt"

COLS = [
    "duration","protocol_type","service","flag","src_bytes","dst_bytes","land",
    "wrong_fragment","urgent","hot","num_failed_logins","logged_in","num_compromised",
    "root_shell","su_attempted","num_root","num_file_creations","num_shells",
    "num_access_files","num_outbound_cmds","is_host_login","is_guest_login","count",
    "srv_count","serror_rate","srv_serror_rate","rerror_rate","srv_rerror_rate",
    "same_srv_rate","diff_srv_rate","srv_diff_host_rate","dst_host_count",
    "dst_host_srv_count","dst_host_same_srv_rate","dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate","dst_host_srv_diff_host_rate","dst_host_serror_rate",
    "dst_host_srv_serror_rate","dst_host_rerror_rate","dst_host_srv_rerror_rate",
    "label","difficulty",
]

train_raw = pd.read_csv(TRAIN_URL, names=COLS)
test_raw  = pd.read_csv(TEST_URL, names=COLS)
print("raw train:", train_raw.shape, " raw test:", test_raw.shape)
train_raw["label"].value_counts().head(10)


In [ ]:

def clean(df):
    df = df.drop(columns=["difficulty"]).copy()
    df["binary_label"] = (df["label"] != "normal").astype(int)
    return df

train_df = clean(train_raw)
test_df  = clean(test_raw)

CAT_COLS = ["protocol_type", "service", "flag"]
encoders = {}
for c in CAT_COLS:
    le = LabelEncoder()
    le.fit(pd.concat([train_df[c], test_df[c]], axis=0))
    train_df[c] = le.transform(train_df[c])
    test_df[c]  = le.transform(test_df[c])
    encoders[c] = le

FEATURE_COLS = [c for c in train_df.columns if c not in ["label", "binary_label"]]

scaler = StandardScaler()
train_df[FEATURE_COLS] = scaler.fit_transform(train_df[FEATURE_COLS])
test_df[FEATURE_COLS]  = scaler.transform(test_df[FEATURE_COLS])

print("features:", len(FEATURE_COLS))
print("train class balance:", train_df["binary_label"].value_counts(normalize=True).round(3).to_dict())
print("test  class balance:", test_df["binary_label"].value_counts(normalize=True).round(3).to_dict())


**Data dictionary (short version):** every feature is a numeric property of a single
network connection — e.g. `duration` (connection length), `src_bytes`/`dst_bytes` (data sent
each way), `count`/`srv_count` (how many connections to the same host/service recently),
and a family of `*_rate` features (fractions of recent connections that were errors, to the
same service, etc.). `protocol_type`, `service`, and `flag` are categorical and have been
label-encoded. All features are standardized (zero mean, unit variance).


## 2. Reserve a secret test slice, then partition into 5 simulated clients

Before anything gets split across the "five banks," we set aside a chunk of rows
(`HOLDOUT_SIZE` below) that **no client ever sees or trains on**. This reservation is
seed-locked and deterministic — the organizers' Kaggle-prep script reserves the *exact same*
rows independently, which is what makes the leaderboard test set genuinely unseen rather
than just another random sample of data you already trained on. The printed fingerprint hash
below should match the organizers' — that's the integrity check.

The *remaining* rows are then split evenly across the 5 simulated "banks."


In [ ]:

import hashlib

N_CLIENTS = 5
HOLDOUT_SIZE = 15000  # rows reserved for the secret Kaggle test set -- never partitioned to clients

_rng = np.random.RandomState(SEED)
_perm = _rng.permutation(len(train_df))
_holdout_idx = _perm[:HOLDOUT_SIZE]
_pool_idx = _perm[HOLDOUT_SIZE:]

fingerprint = hashlib.md5(str(sorted(_holdout_idx.tolist())).encode()).hexdigest()[:12]
print(f"Reserved {len(_holdout_idx)} rows as the secret test slice (never seen by any client).")
print(f"Holdout fingerprint: {fingerprint}  <- this must match the organizers' prep-script output")

train_pool = train_df.iloc[_pool_idx].reset_index(drop=True)
print(f"Remaining pool for client partitioning: {len(train_pool)} rows")


In [ ]:

def make_iid_partition(df, n_clients=N_CLIENTS, seed=SEED):
    rng = np.random.RandomState(seed)
    idx = rng.permutation(len(df))
    chunks = np.array_split(idx, n_clients)
    return [df.iloc[c].reset_index(drop=True) for c in chunks]

iid_clients = make_iid_partition(train_pool)

print("Client sizes:", [len(c) for c in iid_clients])
for i, c in enumerate(iid_clients):
    print(f"  client {i} label mix:", c["binary_label"].value_counts(normalize=True).round(2).to_dict())


## 3. The weak baseline model

This is deliberately small and undertrained — a single 8-unit hidden layer. It works, but
it's mediocre. **This is what today's track improves on.**


In [ ]:

N_FEATURES = len(FEATURE_COLS)

class WeakMLP(nn.Module):
    def __init__(self, n_features=N_FEATURES, hidden=8):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 1),
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)


## 4. The FL harness: local training + FedAvg

This is the plumbing — local training, flattening/unflattening model weights, FedAvg
aggregation, and evaluation. **It already works end to end.** You generally won't need to
touch this section.


In [ ]:

X_test = torch.tensor(test_df[FEATURE_COLS].values, dtype=torch.float32)
y_test = torch.tensor(test_df["binary_label"].values, dtype=torch.float32)

def df_to_tensors(df):
    X = torch.tensor(df[FEATURE_COLS].values, dtype=torch.float32)
    y = torch.tensor(df["binary_label"].values, dtype=torch.float32)
    return X, y

def local_train(model, X, y, epochs=1, lr=0.05, batch_size=256):
    model = copy.deepcopy(model)
    opt = torch.optim.SGD(model.parameters(), lr=lr)
    loss_fn = nn.BCEWithLogitsLoss()
    n = len(X)
    for _ in range(epochs):
        perm = torch.randperm(n)
        for i in range(0, n, batch_size):
            idx = perm[i:i + batch_size]
            opt.zero_grad()
            loss = loss_fn(model(X[idx]), y[idx])
            loss.backward()
            opt.step()
    return model

def get_flat_params(model):
    return torch.cat([p.data.view(-1) for p in model.parameters()])

def set_flat_params(model, flat):
    i = 0
    for p in model.parameters():
        n = p.numel()
        p.data.copy_(flat[i:i + n].view(p.shape))
        i += n

def fedavg(models, weights):
    weights = np.array(weights, dtype=float) / np.sum(weights)
    flats = torch.stack([get_flat_params(m) for m in models])
    avg = (flats * torch.tensor(weights, dtype=torch.float32).unsqueeze(1)).sum(dim=0)
    out = copy.deepcopy(models[0])
    set_flat_params(out, avg)
    return out

def evaluate(model):
    model.eval()
    with torch.no_grad():
        preds = (torch.sigmoid(model(X_test)) > 0.5).float()
    return {
        "precision": round(precision_score(y_test, preds, zero_division=0), 4),
        "recall": round(recall_score(y_test, preds, zero_division=0), 4),
        "f1": round(f1_score(y_test, preds, zero_division=0), 4),
    }

def run_fl(client_dfs, model_fn=lambda: WeakMLP(), rounds=8, epochs=1, verbose=True):
    '''Runs `rounds` communication rounds: every client trains locally starting from the
    current global model, then FedAvg aggregates the updates.'''
    global_model = model_fn()
    client_tensors = [df_to_tensors(df) for df in client_dfs]

    history = []
    for r in range(rounds):
        local_models, sizes = [], []
        for X, y in client_tensors:
            lm = local_train(global_model, X, y, epochs=epochs)
            local_models.append(lm)
            sizes.append(len(X))

        global_model = fedavg(local_models, sizes)

        metrics = evaluate(global_model)
        history.append(metrics)
        if verbose:
            print(f"round {r+1:>2}: {metrics}")

    return global_model, history


## 5. Run the baseline end to end

This is your sanity check. Run this now — it should take under a minute on CPU and land
around **F1 ≈ 0.75–0.78**. That number is your starting point to beat.


In [ ]:

print("Baseline: weak model, 5 clients, FedAvg")
baseline_model, baseline_history = run_fl(iid_clients, rounds=6)
print("\\nFinal:", baseline_history[-1])


---
## 🟢 6. Today's track — build the best *local* detector

The FL loop above is fixed; what you control is what each client trains. Ideas, roughly in
order of effort:
1. **Class imbalance** — attacks are less common than normal traffic. Try class-weighted
   loss (`pos_weight` in `BCEWithLogitsLoss`), oversampling the minority class per client, or
   tuning the decision threshold instead of the fixed 0.5 cutoff.
2. **Feature engineering** — not all 41 features are equally useful; try dropping
   near-constant ones, or engineering ratios/interactions.
3. **A better model** — more hidden units, another layer, dropout, or swap the architecture
   entirely (a small 1D-conv or even a scikit-learn model wrapped to match the interface).

**Your hook:** write a new `model_fn` and/or a modified `local_train`, then re-run `run_fl`.


In [ ]:

# 🔧 YOUR TURN — today's track
# Example skeleton: a slightly less-weak MLP. Replace with your own architecture,
# class-imbalance handling, or feature engineering.

class MyModel(nn.Module):
    def __init__(self, n_features=N_FEATURES, hidden=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, hidden),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(),
            nn.Linear(hidden // 2, 1),
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

my_model, my_history = run_fl(iid_clients, model_fn=lambda: MyModel(), rounds=8)
print("\\nYour result:", my_history[-1], "  (baseline was:", baseline_history[-1], ")")


---
## 7. Leaderboard submission

Run this to score your model on the organizers' Kaggle leaderboard for today's track.


In [ ]:

KAGGLE_TEST_PATH = "test_public.csv"  # adjust path if needed (e.g. /kaggle/input/<comp>/test_public.csv)
FINAL_MODEL = my_model  # <- point this at whichever model you want submitted

kaggle_test = pd.read_csv(KAGGLE_TEST_PATH)
X_kaggle = torch.tensor(kaggle_test[FEATURE_COLS].values, dtype=torch.float32)

FINAL_MODEL.eval()
with torch.no_grad():
    preds = (torch.sigmoid(FINAL_MODEL(X_kaggle)) > 0.5).long().numpy()

submission_df = pd.DataFrame({"Id": kaggle_test["Id"], "Expected": preds})
submission_df.to_csv("submission.csv", index=False)
print(submission_df.head())
print(f"\\nWrote {len(submission_df)} predictions to submission.csv — upload this file to the Beginner Track Leaderboard on Kaggle.")


### What to hand in today
1. Your `submission.csv`, uploaded to the Beginner Track Leaderboard competition.
2. Keep this notebook — Day 2's notebook builds on the same ideas (nothing to redo).

See you tomorrow for the next two tracks.
